# 📖 Notebook 3: Anonymization Techniques

When you need to share or analyze personal data without exposing individual identities, you need **anonymization**. This notebook covers four practical techniques used at companies like Microsoft, Google, and Apple.

## Learning Objectives

By the end of this notebook, you'll understand:
- **k-Anonymity**: making every record look like at least k-1 others
- **l-Diversity**: ensuring sensitive values are diverse within each group
- **Differential Privacy**: adding mathematical noise so individuals can't be identified
- **Tokenization**: replacing PII with reversible tokens (pseudonymization)
- When to use each technique and their trade-offs

## 🛠️ Setup

```bash
cd 08-enterprise/privacy-review
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import hashlib
import random
import math
import uuid
from collections import Counter, defaultdict

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

# Differential privacy is randomised by construction. Seed it so this notebook
# tells the same story twice — and so the assertions further down are not a
# coin flip. Never seed a real DP deployment: predictable noise is no noise.
random.seed(20260821)

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🔍 Let's Load Our Data

First, let's load the user data we'll be anonymizing. In a real scenario, this would be a dataset you want to share with analysts or researchers without exposing individual identities.

In [ ]:
def load_user_data():
    """Load user data that we'll practice anonymizing."""
    conn = get_db_connection()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # The whole table, not a sample. k-anonymity is a property of the release
    # you publish: shrink the release and the equivalence classes shrink with
    # it, so a 20-row "preview" is harder to anonymize than the full 50 rows.
    cursor.execute("""
        SELECT id, first_name, last_name, email, date_of_birth,
               city, state, zip_code, account_status, signup_source, created_at
        FROM users
        ORDER BY id
    """)

    users = [dict(row) for row in cursor.fetchall()]
    for u in users:
        # Signup cohort — the kind of harmless-looking column an analyst asks
        # for. We use it later to show what one extra column costs.
        u["signup_month"] = u["created_at"].strftime("%Y-%m")
    conn.close()
    return users

users = load_user_data()

print(f"📋 Original Data (first 5 of {len(users)} users) — BEFORE anonymization")
print("=" * 100)
print(f"  {'ID':<4} {'Name':<20} {'Email':<25} {'DOB':<12} {'City':<15} {'State':<6} {'ZIP'}")
print("-" * 100)
for u in users[:5]:
    name = f"{u['first_name']} {u['last_name']}"
    dob = u['date_of_birth'].strftime('%Y-%m-%d') if u['date_of_birth'] else 'N/A'
    print(f"  {u['id']:<4} {name:<20} {u['email']:<25} {dob:<12} {u['city']:<15} {u['state']:<6} {u['zip_code']}")

assert len(users) >= 20, (
    f"only {len(users)} users left — notebook 4's purge may have run first; "
    "recreate the database with `docker compose down -v && docker compose up -d`"
)
print(f"\n⚠️  This data contains PII — let's learn how to protect it!")


---
## 🛡️ Technique 1: k-Anonymity

### The Idea

**k-Anonymity** means that for every person in the dataset, there are at least **k-1 other people** who look identical on the quasi-identifier columns (like age, zip code, gender).

**Why?** If you know someone is "34 years old, lives in 98101, male", and only ONE person matches that in the dataset, you've identified them. But if 5 people match (k=5), you can't tell which one they are.

### How It Works

We **generalize** (make less specific) the quasi-identifiers:
- Age 34 → Age range 30-40
- ZIP 98101 → ZIP 981**
- City "Seattle" → State "WA"

We keep generalizing until every combination appears at least k times — and
where generalizing is not enough, we **suppress** (drop) the records that are
still too rare. Generalization alone never gets you there on real data: there
is always a tail of unusual people. Samarati and Sweeney's original definition
pairs the two operations for exactly that reason.

### Two things that are easy to get wrong

1. **k is only k with respect to the quasi-identifiers you declared.** A
   quasi-identifier is *any* attribute the adversary might already know about
   their target — not just the ones you thought to list. Every extra column you
   publish "because analysts want it" joins the QI set whether you declare it
   or not. We measure that cost below.
2. **A table with one class of size 1 is not k-anonymous, no matter what the
   header says.** Always check the smallest class, not the average.


### First: how identifiable is the raw table?

Before applying anything, measure the problem. A "de-identified" release — names
and emails stripped — sounds safe. In 1997 a Massachusetts state agency released
hospital records that way, and Latanya Sweeney re-identified the governor's own
records using nothing but date of birth, ZIP code and sex, joined against a $20
voter roll.

Let's count how many of our users are uniquely identified by their
quasi-identifiers alone.


In [ ]:
# A quasi-identifier is any column an adversary might already know about their
# target. Nobody needs the name column to find someone.
RAW_QUASI_IDENTIFIERS = ("date_of_birth", "zip_code", "city")

raw_classes = Counter(
    tuple(str(u[q]) for q in RAW_QUASI_IDENTIFIERS) for u in users
)
unique_records = sum(count for count in raw_classes.values() if count == 1)

print("🔓 Re-identification Risk in the RAW Data")
print("=" * 70)
print(f"  Quasi-identifiers:      {', '.join(RAW_QUASI_IDENTIFIERS)}")
print(f"  Records:                {len(users)}")
print(f"  Distinct QI combos:     {len(raw_classes)}")
print(f"  Uniquely identifiable:  {unique_records} ({unique_records / len(users):.0%})")

# Play the attacker: we know one person's DOB, ZIP and city.
target = users[3]
knowledge = tuple(str(target[q]) for q in RAW_QUASI_IDENTIFIERS)
matches = [u for u in users if tuple(str(u[q]) for q in RAW_QUASI_IDENTIFIERS) == knowledge]

print(f"\n  🕵️ Attacker knows: DOB={knowledge[0]}, ZIP={knowledge[1]}, city={knowledge[2]}")
print(f"     Matching records: {len(matches)}")
for m in matches:
    print(f"     → {m['first_name']} {m['last_name']}, {m['email']} (user #{m['id']})")

assert unique_records == len(users), (
    "this lab depends on the raw table being fully re-identifiable — if the seed "
    "data changed so that QI combinations repeat, pick sharper quasi-identifiers"
)
assert len(matches) == 1, "the attack must single out exactly one person"
print("\n💡 Every record is unique on three ordinary columns. Deleting the name")
print("   column bought us nothing. That is the problem k-anonymity solves.")


In [ ]:
def generalize_age(dob, level=1):
    """Generalize a date of birth into age ranges.
    Level 1: 5-year ranges (30-34)
    Level 2: 10-year ranges (30-39)
    Level 3: 20-year ranges (20-39)
    """
    if not dob:
        return "unknown"

    from datetime import date
    today = date.today()
    age = today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

    if level == 1:
        bucket = (age // 5) * 5
        return f"{bucket}-{bucket + 4}"
    elif level == 2:
        bucket = (age // 10) * 10
        return f"{bucket}-{bucket + 9}"
    else:
        bucket = (age // 20) * 20
        return f"{bucket}-{bucket + 19}"

def generalize_zip(zip_code, level=1):
    """Generalize a ZIP code by masking digits.
    Level 1: 981** (keep 3 digits)
    Level 2: 98*** (keep 2 digits)
    Level 3: 9**** (keep 1 digit)
    """
    if not zip_code:
        return "*****"

    keep = max(1, 4 - level)
    return zip_code[:keep] + "*" * (len(zip_code) - keep)

def generalize_location(city, state, level=1):
    """Generalize location.
    Level 1: Keep city
    Level 2: State only
    Level 3: Region only
    """
    regions = {
        "WA": "Pacific NW", "OR": "Pacific NW",
        "CA": "West Coast", "NY": "Northeast",
        "TX": "South", "CO": "Mountain",
        "IL": "Midwest", "MA": "Northeast",
        "FL": "Southeast", "GA": "Southeast"
    }

    if level == 1:
        return city
    elif level == 2:
        return state
    else:
        return regions.get(state, "US")

# Demo: show generalization levels
print("📊 Generalization Levels Demo")
print("=" * 65)
u = users[0]

print(f"\n  Original: DOB={u['date_of_birth']}, ZIP={u['zip_code']}, City={u['city']}, State={u['state']}")
for level in [1, 2, 3]:
    age_gen = generalize_age(u['date_of_birth'], level)
    zip_gen = generalize_zip(u['zip_code'], level)
    loc_gen = generalize_location(u['city'], u['state'], level)
    print(f"  Level {level}:  Age={age_gen:<10} ZIP={zip_gen:<8} Location={loc_gen}")

In [ ]:
# The quasi-identifiers we are willing to publish, generalized.
QUASI_IDENTIFIERS = ("age_range", "location", "zip")

def build_release(users, level, extra_fields=()):
    """Generalize every record to `level` and return the table we would publish.

    `extra_fields` are copied through verbatim — we use it below to measure what
    one extra "harmless" analytics column costs.
    """
    release = []
    for u in users:
        record = {
            "age_range": generalize_age(u["date_of_birth"], level),
            "location": generalize_location(u["city"], u["state"], level),
            "zip": generalize_zip(u["zip_code"], level),
            # Not quasi-identifiers in our threat model — the sensitive value
            # (account_status) is what l-Diversity protects further down.
            "account_status": u["account_status"],
            "signup_source": u["signup_source"],
        }
        for field in extra_fields:
            record[field] = u[field]
        release.append(record)
    return release

def equivalence_classes(records, quasi_ids=QUASI_IDENTIFIERS):
    """Group records by their quasi-identifier tuple.

    Every record inside a class is indistinguishable from the others *on those
    columns* — which is exactly as much as k-anonymity ever promises.
    """
    groups = defaultdict(list)
    for record in records:
        groups[tuple(record[q] for q in quasi_ids)].append(record)
    return groups

def apply_k_anonymity(users, k=3, quasi_ids=QUASI_IDENTIFIERS):
    """Generalize, then suppress, until the release really is k-anonymous.

    Generalization on its own does not get there and never did: there is always
    a tail of unusual records that no amount of coarsening merges into a big
    enough class. Those records have to be **suppressed** — dropped from the
    release. Returning a table that still holds a class of size 1 and calling it
    k-anonymous is precisely how "anonymized" datasets get de-anonymized.
    """
    trace = []
    chosen = None

    for level in range(1, 4):
        release = build_release(users, level)
        classes = equivalence_classes(release, quasi_ids)
        too_small = [rows for rows in classes.values() if len(rows) < k]
        trace.append({
            "level": level,
            "classes": len(classes),
            "min_class": min(len(rows) for rows in classes.values()),
            "records_below_k": sum(len(rows) for rows in too_small),
        })
        if not too_small:
            chosen = (level, classes, [])
            break

    if chosen is None:
        # Max generalization still leaves classes below k — suppress them.
        kept = [r for rows in classes.values() if len(rows) >= k for r in rows]
        suppressed = [r for rows in classes.values() if len(rows) < k for r in rows]
        chosen = (3, equivalence_classes(kept, quasi_ids), suppressed)

    level, classes, suppressed = chosen
    return {
        "records": [r for rows in classes.values() for r in rows],
        "classes": classes,
        "level": level,
        "suppressed": suppressed,
        "trace": trace,
        "k": k,
        "quasi_ids": quasi_ids,
    }

# Apply k-anonymity with k=3
k = 3
result = apply_k_anonymity(users, k=k)
anon_data, groups, gen_level = result["records"], result["classes"], result["level"]

print(f"🛡️ k-Anonymity (k={k})")
print("=" * 80)
print("  Generalization search — why we ended up where we did:")
print(f"    {'Level':<7} {'Classes':>8} {'Smallest':>9} {'Records below k':>17}")
for step in result["trace"]:
    verdict = "✅ k reached" if step["records_below_k"] == 0 else "❌ still too rare"
    print(f"    {step['level']:<7} {step['classes']:>8} {step['min_class']:>9} "
          f"{step['records_below_k']:>17}   {verdict}")

if result["suppressed"]:
    rate = len(result["suppressed"]) / len(users)
    print(f"\n  Generalization alone never reached k={k}, so {len(result['suppressed'])} "
          f"records were SUPPRESSED ({rate:.0%} of the table).")
    print("  That is the price of the guarantee — and the number your data team")
    print("  will argue about, because those records are usually the interesting ones.")

print(f"\n📤 Released table: {len(anon_data)} of {len(users)} records, "
      f"generalization level {gen_level}")
print("-" * 80)
print(f"  {'Age Range':<12} {'Location':<15} {'ZIP':<8} {'Status':<12} {'Source'}")
print("-" * 80)
for a in anon_data[:10]:
    print(f"  {a['age_range']:<12} {a['location']:<15} {a['zip']:<8} {a['account_status']:<12} {a['signup_source']}")

print(f"\n📊 Equivalence class sizes in the released table:")
for group_key, rows in sorted(groups.items(), key=lambda x: len(x[1])):
    bar = "█" * min(len(rows), 20)
    print(f"  {str(group_key):<45} {bar} {len(rows)}")

smallest = min(len(rows) for rows in groups.values())
print(f"\n  Smallest equivalence class: {smallest} records → this release is "
      f"{smallest}-anonymous")

# The guarantee, checked. Without this the notebook can print "k-anonymity
# applied" over a table that is nothing of the sort.
assert smallest >= k, (
    f"released table is only {smallest}-anonymous, expected k={k} — suppression "
    "did not remove every under-sized equivalence class"
)
assert len(result["records"]) + len(result["suppressed"]) == len(users), (
    "every input record must be either released or suppressed, never lost"
)
assert len(result["suppressed"]) / len(users) < 0.5, (
    "suppressing half the table is not anonymization, it is deletion — "
    "the generalization ladder is too coarse for this dataset"
)
print(f"✅ k={k} verified on the released table "
      f"({len(result['suppressed'])} records suppressed)")
print(f"\n💡 Names, emails, SSNs and exact DOBs are gone. Only generalized")
print(f"   quasi-identifiers remain — and only for records that hide in a crowd.")


### The column an analyst asks for

Our release is 3-anonymous **with respect to `(age_range, location, zip)`**.
That qualifier is the whole guarantee. An analyst now asks to add
`signup_month` — a cohort column, no names, obviously harmless.

Watch what it does.


In [ ]:
# Same 50 records, same generalization level. The only change is that we
# publish one more column — so both sides of this comparison are like-for-like.
release_plus = build_release(users, result["level"], extra_fields=("signup_month",))
expanded_qi = QUASI_IDENTIFIERS + ("signup_month",)

def release_stats(records, quasi_ids):
    classes = equivalence_classes(records, quasi_ids)
    sizes = [len(rows) for rows in classes.values()]
    return {
        "classes": len(classes),
        "unique": sum(1 for s in sizes if s == 1),
        "must_suppress": sum(s for s in sizes if s < k),
    }

before = release_stats(release_plus, QUASI_IDENTIFIERS)
after = release_stats(release_plus, expanded_qi)

print("🧨 What one extra column costs")
print("=" * 78)
print(f"  {len(users)} records, generalization level {result['level']}, k={k}\n")
print(f"  {'quasi-identifier set':<42} {'classes':>8} {'unique':>7} {'suppress':>9}")
print(f"  {' + '.join(QUASI_IDENTIFIERS):<42} {before['classes']:>8} "
      f"{before['unique']:>7} {before['must_suppress']:>9}")
print(f"  {' + '.join(expanded_qi):<42} {after['classes']:>8} "
      f"{after['unique']:>7} {after['must_suppress']:>9}")

assert after["must_suppress"] > before["must_suppress"] * 3, (
    "adding signup_month should shatter the equivalence classes — if it no "
    "longer does, this demonstration needs a sharper column"
)
print(f"""
💡 The generalization did not change. The data did not change. We published one
   more column, and the number of records we must drop to keep k={k} went from
   {before['must_suppress']} to {after['must_suppress']} — most of the table.

   k-anonymity is a property of the *whole release*, not of the three columns
   you happened to run the check on. Any attribute an adversary could know —
   signup cohort, order count, support-ticket timestamps, an "anonymous" user
   token that also appears in another table — belongs in the quasi-identifier
   set. Re-run the check every time the release changes, and treat every new
   column as a quasi-identifier until you have argued otherwise.
""")


---
## 🌈 Technique 2: l-Diversity

### The Problem with k-Anonymity

k-Anonymity has a weakness: if everyone in a group has the **same sensitive value**, knowing the group reveals the secret.

Example: If all 5 people in the group (age 30-40, Seattle, 981**) have `account_status = 'suspended'`, and you know someone is in that group, you know they're suspended.

### The Fix

**l-Diversity** requires that within each group, the sensitive attribute has at least **l different values**. This prevents the "all the same" attack.

### And the problem with *that*

What we implement below is **distinct l-diversity**, the weakest of the family.
It counts values and ignores how they are spread. A class of 10 people where 9
are `active` and 1 is `deleted` has 2 distinct values and passes 2-diversity —
while an attacker who guesses "active" for anyone in that class is right 90% of
the time. Stronger variants (entropy l-diversity, recursive (c,l)-diversity) put
a floor on the *distribution*, not just the count. So we print the skew next to
the verdict: a green tick with an 80% majority value is not much of a promise.

Note also the ceiling: `account_status` only ever takes three values, so no
amount of generalization can produce 4-diversity on this column. l is bounded by
the number of distinct sensitive values in the whole table.

In [ ]:
def check_l_diversity(records, quasi_ids, sensitive_attr, l=2):
    """Check whether a release satisfies (distinct) l-diversity.

    For each equivalence class, the sensitive attribute must take at least l
    distinct values. We also report the share held by the most common value,
    because "2 distinct values" says nothing about whether one of them covers
    95% of the class.
    """
    results = []
    for group_key, rows in equivalence_classes(records, quasi_ids).items():
        values = [r[sensitive_attr] for r in rows]
        counts = Counter(values)
        results.append({
            "group": group_key,
            "size": len(values),
            "distinct_values": len(counts),
            "top_value": counts.most_common(1)[0][0],
            "top_share": counts.most_common(1)[0][1] / len(values),
            "values": dict(counts),
            "satisfies_l_diversity": len(counts) >= l,
        })
    return sorted(results, key=lambda r: r["group"])

# l-Diversity is checked on the *released* table — the one that survived
# suppression. Checking it on records you are not publishing proves nothing.
sensitive_attr = "account_status"
l = 2

diversity_results = check_l_diversity(anon_data, QUASI_IDENTIFIERS, sensitive_attr, l=l)

print(f"🌈 l-Diversity Check (l={l}, sensitive attribute: '{sensitive_attr}')")
print("=" * 80)

for r in diversity_results:
    icon = "✅" if r["satisfies_l_diversity"] else "❌"
    skew = "⚠️ skewed" if r["top_share"] >= 0.75 else ""
    print(f"\n  {icon} Group: {r['group']}")
    print(f"     Size: {r['size']} records, {r['distinct_values']} distinct values")
    print(f"     Values: {r['values']}")
    print(f"     Most common: '{r['top_value']}' at {r['top_share']:.0%} {skew}")

achieved_l = min(r["distinct_values"] for r in diversity_results)
worst_skew = max(diversity_results, key=lambda r: r["top_share"])
all_satisfy = all(r["satisfies_l_diversity"] for r in diversity_results)

print("\n" + "=" * 80)
print(f"  {'✅' if all_satisfy else '❌'} Release {'satisfies' if all_satisfy else 'does NOT satisfy'} "
      f"{l}-diversity (achieved l = {achieved_l})")
print(f"  ⚠️  Worst class: {worst_skew['group']} — guessing '{worst_skew['top_value']}' "
      f"is right {worst_skew['top_share']:.0%} of the time")

# What would a stricter l cost us?
strict = check_l_diversity(anon_data, QUASI_IDENTIFIERS, sensitive_attr, l=3)
failing = [r for r in strict if not r["satisfies_l_diversity"]]
print(f"\n  At l=3, {len(failing)} of {len(strict)} classes fail — those records would")
print(f"  need further generalization or suppression on top of what k-anonymity")
print(f"  already cost us.")

assert achieved_l >= l, (
    f"released table is only {achieved_l}-diverse on {sensitive_attr}; groups "
    "with a single sensitive value leak that value to anyone who can place a "
    "person in the group"
)
assert worst_skew["top_share"] < 1.0, "a class with one value is not diverse at all"
assert len(failing) > 0, (
    "l=3 is expected to fail here — if it now passes, the point about distinct "
    "l-diversity being cheap to satisfy needs a stricter example"
)
print("\n✅ l-diversity assertions passed")

if not all_satisfy:
    print("\n💡 Fix: Groups with insufficient diversity need further generalization")
    print("   or records must be suppressed (removed) from the dataset.")


---
## 🎲 Technique 3: Differential Privacy

### The Idea

**Differential Privacy** takes a completely different approach: instead of modifying the data itself, we **add random noise** to the answers we compute from the data.

The key insight: if adding or removing any single person from the dataset barely changes the output, then the output doesn't reveal anything about that person.

### The Math (Simplified)

We add noise from a **Laplace distribution**. The amount of noise depends on:
- **Sensitivity**: how much one person can change the answer (usually 1 for counting queries)
- **Epsilon (ε)**: the privacy budget — smaller ε = more noise = more privacy

```
noisy_answer = true_answer + Laplace(0, sensitivity/epsilon)
```

This is what Apple uses for keyboard data, Google uses for Chrome metrics, and the US Census uses for population counts.

### The part everyone skips: composition

ε is not a property of a query. It is a property of **everything you ever
release about the same people**, because differential privacy composes:

- **Sequential composition** — answer k different queries at ε each and you have
  spent k·ε on that dataset.
- **Parallel composition** — queries over *disjoint* groups of people (one count
  per city) cost max(ε), not the sum, because each person's record influences
  exactly one of the answers.

So "we use ε=1" means nothing on its own. ε=1 per query and a thousand queries a
day is ε=1000, which is not a privacy guarantee, it is a rounding error with
paperwork. The number only has meaning next to a **privacy budget**: a total ε
for a dataset, drawn down by every release, and *refused* when it runs out.

That refusal is the whole mechanism. Without it, an attacker just asks the same
question repeatedly and averages the noise away — which is exactly what we
demonstrate below.

In [ ]:
def laplace_noise(sensitivity, epsilon):
    """Generate noise from a Laplace distribution.

    Args:
        sensitivity: how much one person can change the query result
        epsilon: privacy budget spent on this release (smaller = more noise)

    Returns:
        A random noise value
    """
    if epsilon <= 0:
        raise ValueError("epsilon must be positive")

    scale = sensitivity / epsilon
    # Inverse-transform sampling. random.random() can return exactly 0.0, which
    # would put u at the edge and send log(0) to -inf, so keep u strictly inside.
    u = random.random() - 0.5
    u = min(max(u, -0.5 + 1e-12), 0.5 - 1e-12)
    return -scale * math.copysign(1, u) * math.log(1 - 2 * abs(u))


class BudgetExhausted(RuntimeError):
    """Raised when a query would spend more privacy than the dataset has left."""


class PrivacyBudget:
    """The privacy budget for one dataset.

    Every DP release spends ε, and under sequential composition those spends
    add up. Once the total is gone the only honest answer to the next question
    is "no". A DP implementation without one of these is not doing DP — it is
    adding decorative noise.
    """

    def __init__(self, dataset, total_epsilon):
        self.dataset = dataset
        self.total = total_epsilon
        self.spent = 0.0
        self.ledger = []

    @property
    def remaining(self):
        return round(self.total - self.spent, 6)

    def spend(self, epsilon, description):
        if epsilon > self.remaining:
            raise BudgetExhausted(
                f"{self.dataset}: '{description}' needs ε={epsilon}, but only "
                f"ε={self.remaining} of ε={self.total} is left"
            )
        self.spent += epsilon
        self.ledger.append((epsilon, description))
        return epsilon

    def report(self):
        print(f"  💰 Budget for {self.dataset}: spent ε={self.spent:.2f} "
              f"of ε={self.total:.2f} (ε={self.remaining:.2f} left)")
        for eps, desc in self.ledger:
            print(f"     −{eps:<5} {desc}")


def dp_count(true_count, epsilon=1.0):
    """A differentially private count.

    Sensitivity = 1: adding or removing one person changes a count by 1.
    Clamping to >= 0 is post-processing, which DP allows for free.
    """
    return max(0, round(true_count + laplace_noise(sensitivity=1, epsilon=epsilon)))


def dp_histogram(counts, epsilon, budget, description):
    """A DP histogram over disjoint groups (one bucket per city).

    Parallel composition: every user lives in exactly one city, so noising all
    the buckets at ε costs ε *in total*, not ε per bucket.
    """
    budget.spend(epsilon, description)
    return {key: dp_count(value, epsilon) for key, value in counts.items()}


def dp_mean(values, epsilon, lower, upper, budget, description):
    """A DP mean of values clipped to the PUBLIC range [lower, upper].

    The bounds must not be derived from the data. `max(values) - min(values)`
    is itself a private quantity, and using it as the sensitivity leaks the very
    extremes it is meant to hide — pick bounds from the schema or from public
    knowledge, then clip.

    The row count is private too, so we split ε: half on a noisy sum
    (sensitivity = upper − lower after clipping) and half on a noisy count
    (sensitivity = 1).
    """
    budget.spend(epsilon, description)
    clipped = [min(max(float(v), lower), upper) for v in values]
    noisy_sum = sum(clipped) + laplace_noise(upper - lower, epsilon / 2)
    noisy_n = max(1.0, len(clipped) + laplace_noise(1, epsilon / 2))
    return noisy_sum / noisy_n


# Every release below is charged to one budget for the users table.
users_budget = PrivacyBudget("users table", total_epsilon=3.0)

print("🎲 Differential Privacy Demo")
print("=" * 70)

# Query: How many users are in each city?
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT city, COUNT(*) FROM users GROUP BY city ORDER BY COUNT(*) DESC")
city_counts = dict(cursor.fetchall())
cursor.execute("SELECT total FROM orders WHERE total IS NOT NULL")
order_totals = [row[0] for row in cursor.fetchall()]
conn.close()

# Three releases of the same histogram at three privacy levels. Each one is a
# separate release of the same people, so each one costs its own epsilon.
releases = {}
for eps in (1.0, 0.5, 0.1):
    releases[eps] = dp_histogram(city_counts, eps, users_budget,
                                 f"users-per-city histogram at ε={eps}")

print("\n📊 Users per City — True vs Differentially Private Counts")
print(f"  {'City':<18} {'True':>6} {'ε=1.0':>8} {'ε=0.5':>8} {'ε=0.1':>8}")
print("-" * 55)
for city, true_count in city_counts.items():
    print(f"  {city:<18} {true_count:>6} {releases[1.0][city]:>8} "
          f"{releases[0.5][city]:>8} {releases[0.1][city]:>8}")

# A different question about the same people — sequential composition, so this
# one adds to the bill rather than sharing it.
true_mean = sum(float(t) for t in order_totals) / len(order_totals)
private_mean = dp_mean(order_totals, 0.4, lower=0.0, upper=600.0,
                       budget=users_budget,
                       description="mean order total, public bounds [0, 600]")
print(f"\n💵 Mean order total — true ${true_mean:.2f} vs DP ${private_mean:.2f} (ε=0.4)")

print("\n🧾 What that table cost:")
users_budget.report()

print("""
💡 Lower epsilon = more noise = more privacy. ε=1.0 is moderate, ε=0.1 is strong.
   But read the ledger: printing one histogram three ways spent ε=1.6, and the
   mean cost another ε=0.4. Nothing about the *data* changed — we simply told
   the world more about it. That is what composition means, and it is why the
   headline "we use ε=1" is an incomplete sentence.
""")

assert users_budget.spent == 2.0, f"expected ε=2.0 spent, got {users_budget.spent}"
assert abs(private_mean - true_mean) < 100, (
    "the DP mean should still be in the right ballpark at ε=0.4 — a wild answer "
    "means the sensitivity is wrong, not that DP is working"
)
try:
    dp_histogram(city_counts, 2.0, users_budget, "one query too many")
    raise AssertionError("the budget must refuse a query it cannot afford")
except BudgetExhausted as e:
    print(f"🛑 Refused, correctly: {e}")


In [ ]:
# ── The averaging attack, and why the budget exists ────────────────────────
print("🎯 Attack: ask the same question over and over")
print("=" * 70)

true_count = len(users)
epsilon = 0.5

# First, with no accounting at all — the way DP is usually demoed.
unbudgeted = [dp_count(true_count, epsilon) for _ in range(20)]
attack_estimate = sum(unbudgeted) / len(unbudgeted)

print(f"\n  True count: {true_count}")
print(f"  20 answers at ε={epsilon}: {unbudgeted}")
print(f"  Attacker's average: {attack_estimate:.1f} "
      f"(error {abs(attack_estimate - true_count):.1f})")
print(f"""
  The noise cancelled. Averaging k independent answers shrinks the error by
  about √k, so a determined attacker recovers the exact count — and with it,
  the answer to "is my neighbour in this dataset?". This is not a bonus feature
  of DP, it is the attack DP is defined against. Twenty queries at ε=0.5 is a
  total spend of ε={20 * epsilon:.0f}, which guarantees nothing whatsoever.
""")

# Now the same attack against a budget.
attacker_budget = PrivacyBudget("users table (attacker's view)", total_epsilon=2.0)
answers = []
try:
    for i in range(20):
        attacker_budget.spend(epsilon, f"repeat query #{i + 1}")
        answers.append(dp_count(true_count, epsilon))
except BudgetExhausted as e:
    print(f"  🛑 Blocked after {len(answers)} of 20 answers: {e}")

blocked_estimate = sum(answers) / len(answers)
print(f"  Attacker's average from {len(answers)} answers: {blocked_estimate:.1f}")
print(f"  Averaging {len(answers)} draws instead of 20 leaves "
      f"√(20/{len(answers)}) ≈ {math.sqrt(20 / len(answers)):.1f}× more expected error —")
print(f"  and, more to the point, query #{len(answers) + 1} is refused no matter how "
      f"patient the attacker is.")

assert len(answers) == int(attacker_budget.total / epsilon), (
    "the budget must cut the attacker off, not merely count"
)
print("\n✅ The budget — not the noise — is what stops repeated querying.")

# ── Where "DP gets better with more data" is actually true ─────────────────
print("\n\n📈 Why DP works at scale (and not for small groups)")
print("=" * 70)

# Same noise draws applied to three dataset sizes: a paired comparison, so the
# only thing that changes between rows is n.
noise_draws = [abs(laplace_noise(sensitivity=1, epsilon=epsilon)) for _ in range(200)]
mean_abs_error = sum(noise_draws) / len(noise_draws)

print(f"\n  Mean absolute error at ε={epsilon}: {mean_abs_error:.2f} people")
print("  — and that number does NOT depend on the size of the dataset:\n")
print(f"  {'True count':>12} {'Mean abs error':>16} {'Relative error':>16}")

relative_errors = []
for n in (50, 5_000, 500_000):
    relative = mean_abs_error / n
    relative_errors.append(relative)
    print(f"  {n:>12,} {mean_abs_error:>16.2f} {relative:>15.3%}")

assert relative_errors == sorted(relative_errors, reverse=True), (
    "relative error must fall as n grows — that is the entire utility argument "
    "for differential privacy"
)
print(f"""
💡 The Laplace scale is sensitivity/ε — it is fixed. What changes with more data
   is the *ratio*: ±{mean_abs_error:.1f} people is fatal on a count of 50 and invisible on a
   count of 500,000.

   The practical consequence is uncomfortable and worth saying out loud: DP
   protects individuals by making small groups unusable. A count of 3 users in a
   rare category comes back as 0, or 7, or 2. Analysts will ask you to raise ε
   for "just this dashboard". The budget is the record of every time you said
   yes.
""")


---
## 🔑 Technique 4: Tokenization (Pseudonymization)

### The Idea

**Tokenization** replaces PII with random tokens, but **keeps a secure mapping** so you can reverse it when needed. This is **pseudonymization** — the data is still technically personal data (because it's reversible), but it's much safer.

### Why Use It?

- **Internal analytics**: Data engineers work with tokens instead of real emails
- **Cross-system linking**: Join data across databases using tokens without exposing PII
- **Breach impact reduction**: If the analytics DB is breached, attacker only gets tokens
- **GDPR compliance**: Pseudonymization is specifically called out as a safeguard in GDPR

We'll store the token-to-PII mapping in **Redis** (separate from the main database).

In [ ]:
class Tokenizer:
    """Tokenization service: replace PII with tokens, store mapping in Redis."""

    def __init__(self, redis_client, namespace="token"):
        self.r = redis_client
        self.namespace = namespace

    def tokenize(self, value, pii_type="generic"):
        """Replace a PII value with a token.
        If the same value was tokenized before, return the same token (idempotent).
        """
        if not value:
            return None

        # Check if already tokenized
        existing = self.r.get(f"{self.namespace}:value_to_token:{pii_type}:{value}")
        if existing:
            return existing

        # Generate a new token
        token = f"TOK-{pii_type.upper()}-{uuid.uuid4().hex[:12]}"

        # Store bidirectional mapping
        pipe = self.r.pipeline()
        pipe.set(f"{self.namespace}:value_to_token:{pii_type}:{value}", token)
        pipe.set(f"{self.namespace}:token_to_value:{token}", value)
        # Track token metadata
        pipe.hset(f"{self.namespace}:meta:{token}", mapping={
            "pii_type": pii_type,
            "created_at": datetime.now().isoformat()
        })
        pipe.execute()

        return token

    def detokenize(self, token):
        """Recover the original value from a token (requires access to Redis)."""
        return self.r.get(f"{self.namespace}:token_to_value:{token}")

    def delete_mapping(self, token):
        """Permanently delete a token mapping (for GDPR deletion requests)."""
        original = self.r.get(f"{self.namespace}:token_to_value:{token}")
        meta = self.r.hgetall(f"{self.namespace}:meta:{token}")

        if original and meta:
            pii_type = meta.get("pii_type", "generic")
            pipe = self.r.pipeline()
            pipe.delete(f"{self.namespace}:value_to_token:{pii_type}:{original}")
            pipe.delete(f"{self.namespace}:token_to_value:{token}")
            pipe.delete(f"{self.namespace}:meta:{token}")
            pipe.execute()
            return True
        return False

from datetime import datetime

# Create the tokenizer
r = get_redis_client()
tokenizer = Tokenizer(r)

# Tokenize user data
print("🔑 Tokenization Demo")
print("=" * 90)
print(f"  {'Original Email':<30} {'Token':<40} {'Reversible?'}")
print("-" * 90)

tokens_created = []
for u in users[:5]:
    token = tokenizer.tokenize(u["email"], pii_type="email")
    reversed_value = tokenizer.detokenize(token)
    match = "✅ Yes" if reversed_value == u["email"] else "❌ No"
    print(f"  {u['email']:<30} {token:<40} {match}")
    tokens_created.append(token)

# Show idempotency — same email gets same token
print(f"\n💡 Idempotency check:")
same_token = tokenizer.tokenize(users[0]["email"], pii_type="email")
print(f"   Tokenizing '{users[0]['email']}' again → {same_token}")
print(f"   Same as before? {'✅ Yes' if same_token == tokens_created[0] else '❌ No'}")

In [ ]:
# Tokenize an entire user record

def tokenize_user(user, tokenizer):
    """Create a tokenized version of a user record.

    Direct identifiers are replaced with tokens. The remaining columns are the
    ones analysts asked to keep — note that "not a direct identifier" is not the
    same as "not personal data": city, state, signup source and account status
    are quasi-identifiers, and we measure below how well they single people out.
    """
    return {
        "id": tokenizer.tokenize(str(user["id"]), "user_id"),
        "name": tokenizer.tokenize(f"{user['first_name']} {user['last_name']}", "name"),
        "email": tokenizer.tokenize(user["email"], "email"),
        # Quasi-identifiers, kept for analytics
        "city": user["city"],
        "state": user["state"],
        "account_status": user["account_status"],
        "signup_source": user["signup_source"]
    }

print("📋 Tokenized User Records (safe for analytics teams)")
print("=" * 110)

tokenized_users = [tokenize_user(u, tokenizer) for u in users]

for tok in tokenized_users[:5]:
    print(f"\n  User Token: {tok['id']}")
    print(f"    Name:   {tok['name']}")
    print(f"    Email:  {tok['email']}")
    print(f"    City:   {tok['city']}")
    print(f"    Status: {tok['account_status']}")

# How much anonymity do the "non-PII" columns actually leave?
LEFTOVER_QI = ("city", "state", "signup_source", "account_status")
leftover_classes = Counter(tuple(t[q] for q in LEFTOVER_QI) for t in tokenized_users)
singled_out = sum(1 for count in leftover_classes.values() if count == 1)

print(f"\n🔎 Linkage check on the columns we did NOT tokenize")
print(f"   {' + '.join(LEFTOVER_QI)}")
print(f"   {len(leftover_classes)} distinct combinations across {len(tokenized_users)} records")
print(f"   {singled_out} records are uniquely identified by those four columns alone")

assert singled_out > 0, (
    "if nothing is singled out here the point is lost — pick a sharper column set"
)
print(f"""
💡 Analysts can count users per city and analyse signup sources without ever
   seeing a name or an email, and if the analytics store is breached the
   attacker gets tokens. That is a real, worthwhile reduction in blast radius.

   It is not anonymity. {singled_out} of these records are unique on four columns nobody
   thought of as PII, the tokens are stable so every row belonging to one person
   still links together, and anyone with Redis access reverses the whole thing in
   one command. Pseudonymized data is still personal data — GDPR says so
   explicitly (Recital 26), and so does the arithmetic above.
""")


In [ ]:
# GDPR deletion: destroy the token mapping ("crypto-shredding")

print("🗑️ GDPR Deletion Request Demo")
print("=" * 60)

# User requests account deletion
token_to_delete = tokens_created[0]
original_before = tokenizer.detokenize(token_to_delete)
print(f"\n  Token: {token_to_delete}")
print(f"  Before deletion — resolves to: {original_before}")

# Delete the mapping
deleted = tokenizer.delete_mapping(token_to_delete)
print(f"\n  🗑️ Mapping deleted: {deleted}")

# Try to resolve again
original_after = tokenizer.detokenize(token_to_delete)
print(f"  After deletion — resolves to: {original_after}")

assert deleted is True, "delete_mapping must report what it did"
assert original_after is None, "the mapping must really be gone, both directions"
# Both directions must go. If the value→token key survived, the next call to
# tokenize() would hand back the old token and quietly resurrect the link.
forward_key = f"{tokenizer.namespace}:value_to_token:email:{original_before}"
assert tokenizer.r.get(forward_key) is None, (
    "the forward mapping survived the delete — the link is not destroyed"
)

print("""
💡 What just happened, precisely:

   The mapping is destroyed, so nobody — us included — can turn that token back
   into an email address. This is crypto-shredding, and it is a genuine control:
   it is often the only practical way to honour an erasure request against
   append-only stores, backups and log archives.

   What it is NOT is anonymization:
   • The original email is still in the `users` table, in every replica and in
     every backup. We destroyed one copy of the link, not the data.
   • The token is stable, so every analytics row carrying it still links that
     person's records to each other. A behaviour trail is a fingerprint.
   • The rest of the record still singles the person out — the linkage check in
     the previous cell found records that are unique on four "non-PII" columns.

   GDPR Recital 26 asks whether re-identification is possible using "all the
   means reasonably likely to be used", by anyone, not just by you. A deleted
   mapping does not answer that question on its own. Notebook 4 does the other
   half of the job: erasing the source record, and the copies of the PII that
   were duplicated into other tables along the way.
""")


## 📊 Comparison: When to Use Each Technique

| Technique | Reversible? | Use When | Trade-off | What it does *not* protect against |
|-----------|------------|----------|-----------|-----------------------------------|
| **k-Anonymity** | No | Releasing datasets to researchers | Loses precision (ages become ranges) and drops records — we suppressed 12% of ours | Any attribute you left out of the quasi-identifier set; homogeneous groups |
| **l-Diversity** | No | k-anonymity + sensitive attributes | May need further generalization or suppression | Skew — distinct l-diversity passes a class that is 90% one value |
| **Differential Privacy** | No | Aggregate statistics (counts, averages) | Individual queries are noisy; small groups become unusable | Repeated querying, unless a **privacy budget** is enforced across every release |
| **Tokenization** | Yes (with key) | Internal analytics, cross-system joins | Key compromise reveals all PII | Linkage: tokens are stable, so the data is still personal data |

### What Microsoft Uses

- **Windows telemetry**: Differential privacy for usage statistics, with a per-user budget over a time window — the published work reports ε per contribution *and* how often a user may contribute, because one without the other is not a guarantee
- **Azure analytics**: Tokenization for cross-service correlation
- **Research datasets**: k-anonymity + l-diversity for published studies
- **LinkedIn**: Differential privacy for salary insights and hiring analytics

### The honest summary

None of these four make data safe on their own, and this notebook only
demonstrates them on 50 rows of synthetic data with three quasi-identifiers. A
real release needs a threat model (who is the adversary and what do they already
know?), an auxiliary-data review (what other datasets could this be joined
against?), a re-identification test run by someone trying to break it, and a
decision recorded by a person who is accountable for it. The code is the easy
part.

### Next Notebook

In **Notebook 4: Data Retention & Purging**, we'll implement policies that automatically delete data when its retention period expires.